# Discriminator Guidance

Corrige la deriva del muestreo con un discriminador:
$$\varepsilon_{corr} = \varepsilon_\theta - w\,\sqrt{1-\bar\alpha_t}\,\nabla_{z}\log\tfrac{d}{1-d}.$$

w=0 recupera el muestreo original; w>0 empuja hacia lo real (Kim et al., ICML 2023).

## 0. Entorno y Drive

In [ ]:
import torch
from google.colab import drive
drive.mount('/content/drive')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)
if DEVICE != 'cuda':
    print('ATENCION: sin GPU el calculo de FID es inusable. Runtime -> Change runtime type -> GPU.')

## 1. Configuracion

In [ ]:
WORK = '/content/drive/MyDrive/flower-diffusion/latente'
CFG = {
    'work_dir': WORK,
    'oracle_ckpt':   WORK + '/latent_oracle.pt',   # red entrenada
    'latents_cache': WORK + '/latents.pt',          # latentes reales
    'fake_cache':    WORK + '/fake_latents.pt',      # pool de falsos
    'n_fake': 2048, 'gen_batch': 64, 'gen_steps': 50,
    'disc': {'epochs': 25, 'batch': 128, 'lr': 2e-4, 'ckpt': WORK + '/discriminador.pt'},
}

## 2. Modelo, difusion y carga de la red congelada

In [ ]:
# [IA] Carga de la red congelada (checkpoint EMA)
# y de los latentes reales cacheados, con apoyo de IA.

import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class TimeEmbedding(nn.Module):
    def __init__(self, time_emb_dim):
        super().__init__()
        self.linear1 = nn.Linear(time_emb_dim, time_emb_dim * 4)
        self.linear2 = nn.Linear(time_emb_dim * 4, time_emb_dim)
        self.act = nn.SiLU()
    def get_sinusoidal_embedding(self, t, dim):
        half = dim // 2
        emb = math.log(10000) / (half - 1)
        emb = torch.exp(torch.arange(half, device=t.device) * -emb)
        emb = t.float().unsqueeze(1) * emb.unsqueeze(0)
        return torch.cat([torch.sin(emb), torch.cos(emb)], dim=1)
    def forward(self, t, dim):
        x = self.get_sinusoidal_embedding(t, dim)
        x = self.act(self.linear1(x))
        return self.linear2(x)

class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_emb_dim):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.norm1 = nn.GroupNorm(8, out_ch)
        self.norm2 = nn.GroupNorm(8, out_ch)
        self.act = nn.SiLU()
        self.time_proj = nn.Linear(time_emb_dim, out_ch)
        self.residual = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    def forward(self, x, t_emb):
        h = self.act(self.norm1(self.conv1(x)))
        t = self.act(self.time_proj(t_emb))
        h = h + t[:, :, None, None]
        h = self.act(self.norm2(self.conv2(h)))
        return h + self.residual(x)

class Downsample(nn.Module):
    def __init__(self, ch):
        super().__init__(); self.op = nn.Conv2d(ch, ch, 3, stride=2, padding=1)
    def forward(self, x): return self.op(x)

class Upsample(nn.Module):
    def __init__(self, ch):
        super().__init__(); self.conv = nn.Conv2d(ch, ch, 3, padding=1)
    def forward(self, x): return self.conv(F.interpolate(x, scale_factor=2, mode='nearest'))

class AttentionBlock(nn.Module):
    def __init__(self, channels, num_heads=4):
        super().__init__()
        assert channels % num_heads == 0
        self.num_heads = num_heads
        self.norm = nn.GroupNorm(8, channels)
        self.qkv = nn.Conv2d(channels, channels * 3, 1)
        self.proj = nn.Conv2d(channels, channels, 1)
    def forward(self, x):
        B, C, H, W = x.shape
        q, k, v = self.qkv(self.norm(x)).chunk(3, dim=1)
        dh = C // self.num_heads
        q = q.reshape(B, self.num_heads, dh, H * W)
        k = k.reshape(B, self.num_heads, dh, H * W)
        v = v.reshape(B, self.num_heads, dh, H * W)
        attn = torch.softmax(torch.einsum('bhcn,bhcm->bhnm', q, k) / math.sqrt(dh), dim=-1)
        out = torch.einsum('bhnm,bhcm->bhcn', attn, v).reshape(B, C, H, W)
        return x + self.proj(out)

class UNet(nn.Module):
    def __init__(self, in_channels=4, base_channels=128, channel_mults=(1, 2, 4),
                 num_res_blocks=2, attn_resolutions=(16,), time_emb_dim=128, image_size=32):
        super().__init__()
        self.time_emb_dim = time_emb_dim
        self.time_embedding = TimeEmbedding(time_emb_dim)
        self.init_conv = nn.Conv2d(in_channels, base_channels, 3, padding=1)
        self.downs = nn.ModuleList(); chs = [base_channels]; now = base_channels; res = image_size
        for i, m in enumerate(channel_mults):
            out = base_channels * m
            for _ in range(num_res_blocks):
                blk = nn.ModuleList([ResBlock(now, out, time_emb_dim)]); now = out
                if res in attn_resolutions: blk.append(AttentionBlock(now))
                self.downs.append(blk); chs.append(now)
            if i != len(channel_mults) - 1:
                self.downs.append(nn.ModuleList([Downsample(now)])); chs.append(now); res //= 2
        self.mid_block1 = ResBlock(now, now, time_emb_dim)
        self.mid_attn = AttentionBlock(now)
        self.mid_block2 = ResBlock(now, now, time_emb_dim)
        self.ups = nn.ModuleList()
        for i, m in reversed(list(enumerate(channel_mults))):
            out = base_channels * m
            for _ in range(num_res_blocks + 1):
                blk = nn.ModuleList([ResBlock(now + chs.pop(), out, time_emb_dim)]); now = out
                if res in attn_resolutions: blk.append(AttentionBlock(now))
                self.ups.append(blk)
            if i != 0:
                self.ups.append(nn.ModuleList([Upsample(now)])); res *= 2
        self.out_norm = nn.GroupNorm(8, now); self.out_act = nn.SiLU()
        self.out_conv = nn.Conv2d(now, in_channels, 3, padding=1)
    def forward(self, x, t):
        t_emb = self.time_embedding(t, dim=self.time_emb_dim)
        h = self.init_conv(x); skips = [h]
        for blk in self.downs:
            if isinstance(blk[0], Downsample): h = blk[0](h)
            else:
                h = blk[0](h, t_emb)
                if len(blk) > 1: h = blk[1](h)
            skips.append(h)
        h = self.mid_block1(h, t_emb); h = self.mid_attn(h); h = self.mid_block2(h, t_emb)
        for blk in self.ups:
            if isinstance(blk[0], Upsample): h = blk[0](h)
            else:
                h = torch.cat([h, skips.pop()], dim=1)
                h = blk[0](h, t_emb)
                if len(blk) > 1: h = blk[1](h)
        return self.out_conv(self.out_act(self.out_norm(h)))

class DiffusionProcess:
    def __init__(self, T=1000, beta_start=1e-4, beta_end=0.02, device='cuda'):
        self.T = T; self.device = device
        self.betas = torch.linspace(beta_start, beta_end, T, device=device)
        self.alphas = 1.0 - self.betas
        self.alpha_bars = torch.cumprod(self.alphas, dim=0)
    def add_noise(self, x0, t):
        eps = torch.randn_like(x0)
        ab = self.alpha_bars[t].view(-1, 1, 1, 1)
        return torch.sqrt(ab) * x0 + torch.sqrt(1 - ab) * eps, eps

In [ ]:
def build_latent_model(cfg, device):
    return UNet(in_channels=cfg['latent_ch'], base_channels=cfg['base_channels'],
                channel_mults=tuple(cfg['channel_mults']), num_res_blocks=cfg['num_res_blocks'],
                attn_resolutions=tuple(cfg['attn_resolutions']), time_emb_dim=cfg['time_emb_dim'],
                image_size=cfg['latent_size']).to(device)

ckpt = torch.load(CFG['oracle_ckpt'], map_location=DEVICE)
cfg = ckpt['config']
oracle = build_latent_model(cfg, DEVICE)
oracle.load_state_dict(ckpt['ema_state_dict'])
oracle.eval()
for p in oracle.parameters(): p.requires_grad_(False)   # red congelada
diffusion = DiffusionProcess(cfg['T'], cfg['beta_start'], cfg['beta_end'], DEVICE)

reales = torch.load(CFG['latents_cache'])   # latentes reales cacheados
print('red cargada | latentes reales:', tuple(reales.shape))

## 3. Muestreadores (base y guiado)

`eta_sample_latent`: muestreo normal (solo eta). `eta_sample_guiado`: agrega la corrección del discriminador (parámetro w).

In [ ]:
import numpy as np

@torch.no_grad()
def eta_sample_latent(model, diffusion, n, steps, eta, cfg, device=DEVICE, seed=None):
    if seed is not None: torch.manual_seed(seed)
    model.eval(); T = diffusion.T
    ts = np.linspace(0, T - 1, steps + 1).round().astype(int)[::-1]
    x = torch.randn(n, cfg['latent_ch'], cfg['latent_size'], cfg['latent_size'], device=device)
    for i in range(steps):
        t, tn = int(ts[i]), int(ts[i + 1])
        ab_t = diffusion.alpha_bars[t]; ab_n = diffusion.alpha_bars[tn]
        tb = torch.full((n,), t, device=device, dtype=torch.long)
        eps = model(x, tb)
        x0 = (x - torch.sqrt(1 - ab_t) * eps) / torch.sqrt(ab_t)
        sigma = eta * torch.sqrt((1 - ab_n) / (1 - ab_t)) * torch.sqrt(1 - ab_t / ab_n)
        c = torch.sqrt(torch.clamp(1 - ab_n - sigma ** 2, min=0.0))
        z = torch.randn_like(x) if i < steps - 1 else torch.zeros_like(x)
        x = torch.sqrt(ab_n) * x0 + c * eps + sigma * z
    return x


@torch.no_grad()
def eta_sample_guiado(model, disc, diffusion, n, steps, eta, w, cfg, device=DEVICE, seed=None):
    # muestreo con eta (ruido) y w (guia del discriminador). w=0 recupera el muestreo original.
    if seed is not None: torch.manual_seed(seed)
    model.eval(); disc.eval(); T = diffusion.T
    ts = np.linspace(0, T - 1, steps + 1).round().astype(int)[::-1]
    x = torch.randn(n, cfg['latent_ch'], cfg['latent_size'], cfg['latent_size'], device=device)
    for i in range(steps):
        t, tn = int(ts[i]), int(ts[i + 1])
        ab_t = diffusion.alpha_bars[t]; ab_n = diffusion.alpha_bars[tn]
        tb = torch.full((n,), t, device=device, dtype=torch.long)
        if w != 0.0:
            with torch.enable_grad():
                xg = x.detach().requires_grad_(True)
                logit = disc(xg, tb).sum()
                grad = torch.autograd.grad(logit, xg)[0]
        else:
            grad = torch.zeros_like(x)
        eps = model(x, tb)
        eps = eps - w * torch.sqrt(1 - ab_t) * grad     # correccion del discriminador
        x0 = (x - torch.sqrt(1 - ab_t) * eps) / torch.sqrt(ab_t)
        sigma = eta * torch.sqrt((1 - ab_n) / (1 - ab_t)) * torch.sqrt(1 - ab_t / ab_n)
        c = torch.sqrt(torch.clamp(1 - ab_n - sigma ** 2, min=0.0))
        z = torch.randn_like(x) if i < steps - 1 else torch.zeros_like(x)
        x = torch.sqrt(ab_n) * x0 + c * eps + sigma * z
    return x

## 4. VAE para decodificar

In [ ]:
# [IA] Carga del VAE para decodificar, con apoyo de IA.

!pip -q uninstall -y torchao
!pip -q install -U diffusers safetensors
from diffusers import AutoencoderKL
vae = AutoencoderKL.from_pretrained('stabilityai/sd-vae-ft-mse').to(DEVICE).eval()
for p in vae.parameters(): p.requires_grad_(False)
SCALE = vae.config.scaling_factor
@torch.no_grad()
def decode(z): return vae.decode(z / SCALE).sample.clamp(-1, 1)
print('VAE cargado')

## 5. Discriminador: arquitectura

In [ ]:
class Discriminator(nn.Module):
    # recibe un latente ruidoso z_t (4x32x32) y el paso t; devuelve un logit = log(d/(1-d)),
    # con d = probabilidad de que sea REAL. Su gradiente respecto a z_t es la correccion que guia.
    def __init__(self, in_ch=4, base=64, t_dim=128):
        super().__init__()
        self.t_dim = t_dim
        self.t_mlp = nn.Sequential(nn.Linear(t_dim, t_dim), nn.SiLU(), nn.Linear(t_dim, t_dim))
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, base, 4, 2, 1), nn.SiLU(),                                    # 32 -> 16
            nn.Conv2d(base, base * 2, 4, 2, 1), nn.GroupNorm(8, base * 2), nn.SiLU(),       # 16 -> 8
            nn.Conv2d(base * 2, base * 4, 4, 2, 1), nn.GroupNorm(8, base * 4), nn.SiLU(),   # 8 -> 4
        )
        self.head = nn.Sequential(nn.Linear(base * 4 * 4 * 4 + t_dim, 256), nn.SiLU(), nn.Linear(256, 1))
    def sinus(self, t):
        half = self.t_dim // 2
        f = math.log(10000) / (half - 1)
        f = torch.exp(torch.arange(half, device=t.device) * -f)
        a = t.float().unsqueeze(1) * f.unsqueeze(0)
        return torch.cat([torch.sin(a), torch.cos(a)], dim=1)
    def forward(self, x, t):
        te = self.t_mlp(self.sinus(t))
        h = self.net(x).flatten(1)
        h = torch.cat([h, te], dim=1)
        return self.head(h).squeeze(1)

## 6. Pool de latentes falsos

Latentes generados por la red congelada (cacheados). Son los 'falsos' que el discriminador aprende a distinguir de los reales.

In [ ]:
# [IA] Caché del pool de falsos: generar una vez
# y guardar/recargar, con apoyo de IA.

import os

pool_path = CFG['fake_cache']
if os.path.exists(pool_path):
    falsos = torch.load(pool_path)
    print('pool falso cargado:', tuple(falsos.shape))
else:
    outs = []; hecho = 0
    while hecho < CFG['n_fake']:
        b = min(CFG['gen_batch'], CFG['n_fake'] - hecho)
        z = eta_sample_latent(oracle, diffusion, b, CFG['gen_steps'], eta=0.0, cfg=cfg, device=DEVICE)
        outs.append(z.cpu()); hecho += b
        print(f'  generados {hecho}/{CFG["n_fake"]}')
    falsos = torch.cat(outs, 0)
    torch.save(falsos, pool_path)
    print('pool falso generado y guardado:', tuple(falsos.shape))

## 7. Entrenar el discriminador

Cada paso: un lote de reales y uno de falsos, ruido a un nivel t aleatorio, y aprende a clasificarlos.

In [ ]:
from torch.optim import Adam

def ruido_a_nivel(z0, t):
    ab = diffusion.alpha_bars[t].view(-1, 1, 1, 1)
    return torch.sqrt(ab) * z0 + torch.sqrt(1 - ab) * torch.randn_like(z0)

def entrenar_discriminador(cfg_d):
    disc = Discriminator(in_ch=cfg['latent_ch']).to(DEVICE)
    opt = Adam(disc.parameters(), lr=cfg_d['lr'])
    bce = nn.BCEWithLogitsLoss()
    Nr, Nf = reales.shape[0], falsos.shape[0]
    B = cfg_d['batch']
    pasos_por_epoca = min(Nr, Nf) // B
    for epoch in range(cfg_d['epochs']):
        perm_r = torch.randperm(Nr); perm_f = torch.randperm(Nf)
        perdida = 0.0; acc = 0.0
        for k in range(pasos_por_epoca):
            ir = perm_r[k * B:(k + 1) * B]; if_ = perm_f[k * B:(k + 1) * B]
            zr = reales[ir].to(DEVICE); zf = falsos[if_].to(DEVICE)
            t = torch.randint(0, cfg['T'], (B,), device=DEVICE)
            zr_t = ruido_a_nivel(zr, t); zf_t = ruido_a_nivel(zf, t)
            lr_ = disc(zr_t, t); lf_ = disc(zf_t, t)
            loss = bce(lr_, torch.ones_like(lr_)) + bce(lf_, torch.zeros_like(lf_))
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            perdida += loss.item()
            acc += ((lr_ > 0).float().mean().item() + (lf_ < 0).float().mean().item()) / 2
        print(f"epoca {epoch+1}/{cfg_d['epochs']}  loss {perdida/pasos_por_epoca:.3f}  acc {acc/pasos_por_epoca:.2f}")
    torch.save(disc.state_dict(), cfg_d['ckpt'])
    return disc

# [IA] Atajo para cargar el discriminador ya entrenado en vez de re-entrenarlo
# (load_state_dict desde el checkpoint): sugerido con apoyo de IA.
# disc = Discriminator(in_ch=cfg['latent_ch']).to(DEVICE)
# disc.load_state_dict(torch.load(CFG['disc']['ckpt'], map_location=DEVICE)); disc.eval()
disc = entrenar_discriminador(CFG['disc'])

## 8. Infraestructura de FID

In [ ]:
# [IA] Montaje del FID: carga de Inception
# y estadísticas de las flores reales, con apoyo de IA.

!pip -q install pytorch-fid
import time
from PIL import Image
from pytorch_fid.inception import InceptionV3
from pytorch_fid.fid_score import compute_statistics_of_path, calculate_frechet_distance

SRC_REAL = '/content/drive/MyDrive/flower-diffusion/flowers'
REAL_DIR = '/content/real_256'
GEN_BATCH_FID = 32

# reales redimensionadas a 256x256 (una vez)
os.makedirs(REAL_DIR, exist_ok=True)
files = [f for f in os.listdir(SRC_REAL) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
if len(os.listdir(REAL_DIR)) < len(files):
    for f in files:
        Image.open(os.path.join(SRC_REAL, f)).convert('RGB').resize((256, 256), Image.LANCZOS)\
             .save(os.path.join(REAL_DIR, os.path.splitext(f)[0] + '.png'))
print('reales redimensionadas:', len(os.listdir(REAL_DIR)))

block = InceptionV3.BLOCK_INDEX_BY_DIM[2048]
inception = InceptionV3([block]).to(DEVICE).eval()
print('Estadisticas Inception de las flores reales (una vez)...')
m_real, s_real = compute_statistics_of_path(REAL_DIR, inception, 50, 2048, DEVICE)

def guardar_pngs(imgs, out_dir, start):
    os.makedirs(out_dir, exist_ok=True)
    arr = (imgs * 255).clip(0, 255).astype('uint8')
    for j in range(arr.shape[0]):
        Image.fromarray(arr[j].transpose(1, 2, 0)).save(os.path.join(out_dir, f'{start + j:05d}.png'))

@torch.no_grad()
def fid_de(eta, w, steps=50, M=256):
    # FID de las muestras generadas con (eta, w, steps), usando M muestras. Unica funcion de FID.
    gd = '/content/gen_fid'; os.system(f'rm -rf {gd}'); os.makedirs(gd, exist_ok=True)
    hecho = 0
    while hecho < M:
        b = min(GEN_BATCH_FID, M - hecho)
        z = eta_sample_guiado(oracle, disc, diffusion, b, steps=steps, eta=eta, w=w, cfg=cfg)
        imgs = ((decode(z).cpu() + 1) / 2).clamp(0, 1).numpy()
        guardar_pngs(imgs, gd, hecho); hecho += b
    mg, sg = compute_statistics_of_path(gd, inception, 50, 2048, DEVICE)
    return calculate_frechet_distance(m_real, s_real, mg, sg)

print('FID listo. Prueba: fid_de(0.0, 0.0) deberia dar ~97 (M=256).')

## 9. Cruce η×w (heatmap)


In [ ]:
ETAS = [0.0, 0.5, 1.0]
WS = [0.0, 4.0, 8.0]
M_HEAT = 256

grid = np.zeros((len(ETAS), len(WS)))
for ie, eta in enumerate(ETAS):
    for iw, w in enumerate(WS):
        t0 = time.time()
        grid[ie, iw] = fid_de(eta, w, steps=50, M=M_HEAT)
        print(f'eta={eta}  w={w}  FID={grid[ie, iw]:7.2f}   ({time.time()-t0:.0f}s)')

fig, ax = plt.subplots(figsize=(7, 5.5))
im = ax.imshow(grid, cmap='viridis_r', aspect='auto')
ax.set_xticks(range(len(WS))); ax.set_xticklabels(WS); ax.set_xlabel('w (guia del discriminador)')
ax.set_yticks(range(len(ETAS))); ax.set_yticklabels(ETAS); ax.set_ylabel('eta (ruido del muestreo)')
for ie in range(len(ETAS)):
    for iw in range(len(WS)):
        ax.text(iw, ie, f'{grid[ie, iw]:.1f}', ha='center', va='center', color='white', fontweight='bold')
plt.colorbar(im, label='FID (menor = mejor)')
ax.set_title('Cruce de los dos parametros: FID(eta, w)')
plt.tight_layout()
plt.savefig(CFG['work_dir'] + '/fid_eta_w.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. FID vs η

Barrido de η con w=0 (sin guía): aísla el efecto del azar.

In [ ]:
import matplotlib.pyplot as plt

ETAS_C = list(np.round(np.arange(0.0, 1.001, 0.1), 2))
M_ETA = 256

fids_eta = []
for eta in ETAS_C:
    f = fid_de(eta, 0.0, steps=50, M=M_ETA)
    fids_eta.append(f)
    print(f'  eta={eta:.1f}  FID={f:.2f}')

fig, ax = plt.subplots(figsize=(7.5, 5))
ax.plot(ETAS_C, fids_eta, 'o-', color='tab:blue', lw=2.5, ms=8)
ax.set_xlabel('η   (azar del muestreo)', fontsize=13)
ax.set_ylabel('FID   (menor = mejor)', fontsize=13)
ax.set_title(f'El azar mejora la calidad\nmas η -> menor FID   (w=0, M={M_ETA})', fontsize=15, fontweight='bold')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(CFG['work_dir'] + '/fid_vs_eta.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. FID vs NFE

Varía el número de pasos (NFE) para w=0, 4, 8 con η=0.

In [ ]:
NFES = [5, 10, 20, 50]
WS_NFE = [0.0, 4.0, 8.0]
M_NFE = 256

curvas = {}
for w in WS_NFE:
    fids_w = []
    for nfe in NFES:
        t0 = time.time(); f = fid_de(0.0, w, steps=nfe, M=M_NFE); fids_w.append(f)
        print(f'w={w:<4} NFE={nfe:<3} FID={f:7.2f}   ({time.time()-t0:.0f}s)')
    curvas[w] = fids_w

fig, ax = plt.subplots(figsize=(7, 5))
for w in WS_NFE:
    ax.plot(NFES, curvas[w], 'o-', label=f'w={w}')
ax.set_xlabel('NFE (numero de pasos del muestreo)'); ax.set_ylabel('FID (menor = mejor)')
ax.set_title('FID vs NFE por nivel de guia w (eta=0)')
ax.legend(); plt.tight_layout()
plt.savefig(CFG['work_dir'] + '/fid_nfe_w.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Material visual: barrido de w

In [ ]:
PESOS = [0.0, 2.0, 4.0, 8.0]
N = 6
filas = []
for w in PESOS:
    z = eta_sample_guiado(oracle, disc, diffusion, N, steps=50, eta=0.0, w=w, cfg=cfg, seed=0)
    imgs = ((decode(z).cpu() + 1) / 2).clamp(0, 1).numpy()
    filas.append(imgs)

fig, axes = plt.subplots(len(PESOS), N, figsize=(2.2 * N, 2.2 * len(PESOS)))
for r, w in enumerate(PESOS):
    for cidx in range(N):
        axes[r, cidx].imshow(filas[r][cidx].transpose(1, 2, 0)); axes[r, cidx].axis('off')
    axes[r, 0].set_ylabel(f'w={w}', rotation=0, labelpad=28, va='center', fontsize=11)
fig.suptitle('Discriminator guidance: misma semilla, distinto w (eta=0)')
plt.tight_layout()
plt.savefig(CFG['work_dir'] + '/disc_guidance_barrido_w.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. Cherry-pick con el discriminador

Genera un pool con guía y se queda con las mejor rankeadas por el discriminador.

In [ ]:
N_POOL = 256
W_OPT = 4.0
ETA_OPT = 0.0
TOPK = 16

pool = []; hecho = 0
while hecho < N_POOL:
    b = min(32, N_POOL - hecho)
    z = eta_sample_guiado(oracle, disc, diffusion, b, steps=50, eta=ETA_OPT, w=W_OPT, cfg=cfg)
    pool.append(z.cpu()); hecho += b
    print(f'  generadas {hecho}/{N_POOL}')
pool = torch.cat(pool, 0)

@torch.no_grad()
def puntuar(latents, t_score=50, batch=64):
    scores = []
    for i in range(0, len(latents), batch):
        zb = latents[i:i + batch].to(DEVICE)
        t = torch.full((zb.shape[0],), t_score, device=DEVICE, dtype=torch.long)
        ab = diffusion.alpha_bars[t].view(-1, 1, 1, 1)
        zt = torch.sqrt(ab) * zb + torch.sqrt(1 - ab) * torch.randn_like(zb)
        scores.append(disc(zt, t).cpu())
    return torch.cat(scores, 0)

scores = puntuar(pool)
orden = torch.argsort(scores, descending=True)
top = pool[orden[:TOPK]]
torch.save(top, CFG['work_dir'] + '/top_flores.pt')

imgs = ((decode(top.to(DEVICE)).cpu() + 1) / 2).clamp(0, 1).numpy()
fig, axes = plt.subplots(2, 8, figsize=(18, 4.6))
for ax, im in zip(axes.ravel(), imgs):
    ax.imshow(im.transpose(1, 2, 0)); ax.axis('off')
fig.suptitle(f'Top-{TOPK} flores (guia w={W_OPT}) rankeadas por el discriminador')
plt.tight_layout()
plt.savefig(CFG['work_dir'] + '/top_flores.png', dpi=150, bbox_inches='tight')
plt.show()
print('guardadas en top_flores.pt / top_flores.png')